# 1.04 — Presentation Analytics

**Author:** Yashaswi Mohanty  
**Date:** 2026-03-28  
**Purpose:** Produce the three pending calculations for the Monday presentation.

1. **Petitioner frequency distribution table** — how many petitioners file 1, 2, 3, … complaints (excl. discarded)
2. **Repeat-filer subcategory analysis** — for petitioners who file ≥2 complaints, do they re-file the same subcategory (resolution failure) or different subcategories (active citizen)?
3. **Disposed-only resolution time histogram** — distribution of disposal days for `status = 'Disposed'` across all four fiscal years
4. **Rural housing regression** — OLS of disposal time on district FE vs block FE, restricted to Rural Housing + PMAY subcategories

All analysis uses complaints excluding `status = 'Discard'` unless otherwise noted.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path('__file__').resolve().parent.parent
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import warnings
warnings.filterwarnings('ignore')

import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import statsmodels.formula.api as smf
from IPython.display import display

from app.config import load_duckdb, directories

# ── DPIC / UChicago style ────────────────────────────────────────────────────
MAROON       = '#800000'   # UChicago maroon — primary chart colour
MAROON_MID   = '#A84040'   # mid-tone for secondary elements
MAROON_LIGHT = '#D4A0A0'   # light for annotations / secondary series
NEAR_BLACK   = '#1A1A1A'   # primary text
GREY_AXIS    = '#AAAAAA'   # spine / tick colour
GREY_GRID    = '#E8E8E8'   # grid line colour

# Four-year palette: light → dark maroon (oldest → most recent)
FY_COLORS = ['#D4A0A0', '#B05050', '#8C2E2E', MAROON]

plt.rcParams.update({
    'figure.facecolor':    'white',
    'axes.facecolor':      'white',
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'axes.edgecolor':      GREY_AXIS,
    'axes.grid':           True,
    'axes.grid.axis':      'y',
    'grid.color':          GREY_GRID,
    'grid.linewidth':      0.6,
    'font.family':         'sans-serif',
    'font.size':           10,
    'axes.titlesize':      11,
    'axes.titleweight':    'bold',
    'axes.titlecolor':     NEAR_BLACK,
    'axes.labelcolor':     NEAR_BLACK,
    'xtick.color':         '#555555',
    'ytick.color':         '#555555',
    'xtick.labelsize':     9,
    'ytick.labelsize':     9,
})

OUTPUT = directories.OUTPUT
print(f'Output dir: {OUTPUT}')


## 0. Load and prepare data

In [ ]:
# Try duckdb loader first (works in full uv env); fall back to sqlite3 if
# duckdb extension download is unavailable (e.g. air-gapped / no network).
try:
    df_raw = load_duckdb(output_format="polars")
    print("Loaded via DuckDB")
except Exception as e:
    print(f"DuckDB unavailable ({e}), falling back to sqlite3")
    import sqlite3
    _conn = sqlite3.connect(str(directories.RAW_DATA / "grievance.db"))
    df_raw = pl.read_database("SELECT * FROM complaints", _conn)
    _conn.close()

print(f"Raw rows: {len(df_raw):,}")
print(f"Columns: {df_raw.columns}")

In [ ]:
# ── Cast dates (no-op if already Datetime, handles string fallback) ───────────
# duckdb sqlite_scanner infers DATETIME columns correctly; direct sqlite3 load
# returns them as strings. cast() covers both.

df = (
    df_raw
    .with_columns([
        pl.col("created_on").cast(pl.Datetime, strict=False),
        pl.col("resolved_on").cast(pl.Datetime, strict=False),
        (pl.col("petitioner_name") + "_" + pl.col("petitioner_mobile")).alias("petitioner_id"),
    ])
    .with_columns([
        pl.col("created_on").dt.year().alias("year"),
        pl.col("created_on").dt.month().alias("month"),
        pl.when(pl.col("created_on").dt.month() >= 4)
          .then(pl.col("created_on").dt.year())
          .otherwise(pl.col("created_on").dt.year() - 1)
          .alias("fy_start"),
    ])
    .with_columns(
        (pl.col("fy_start").cast(pl.Utf8) + "-" +
         (pl.col("fy_start") + 1).cast(pl.Utf8).str.slice(2, 2)).alias("fiscal_year")
    )
)

print(df.group_by("status").len().sort("len", descending=True))
print(f"\nFiscal years in data:")
print(df.group_by("fiscal_year").len().sort("fiscal_year"))

In [ ]:
# ── Working dataset: exclude discarded ───────────────────────────────────────
DISCARD_STATUS = "Discard"
RURAL_HOUSING_SUBCATS = ["Rural Housing", "IAY/MKY/BPGY/PMAY"]

df_clean = df.filter(pl.col("status") != DISCARD_STATUS)
print(f"After excluding discards: {len(df_clean):,} rows (dropped {len(df_raw)-len(df_clean):,})")

# Focus FYs for maps / primary analysis
PRIMARY_FYS = ["2021-22", "2022-23", "2023-24", "2024-25"]

---
## 1. Petitioner frequency distribution table

For each petitioner (identified by `name_mobile`), count how many complaints they filed (excl. discarded).  
Output: table of `n_complaints → n_petitioners → share_of_petitioners → cumulative_share`.

In [ ]:
# ── Complaints per petitioner ─────────────────────────────────────────────────
from datetime import datetime

MAX_EXPLICIT = 10  # show rows 1-10 individually; group 11+ together

def complaints_per_petitioner(frame):
    return (
        frame
        .group_by("petitioner_id")
        .agg(pl.len().alias("n_complaints"))
    )

def build_frequency_table(petitioner_counts, max_explicit=MAX_EXPLICIT):
    total_petitioners = len(petitioner_counts)
    freq = (
        petitioner_counts
        .with_columns(
            pl.when(pl.col("n_complaints") <= max_explicit)
              .then(pl.col("n_complaints").cast(pl.Utf8))
              .otherwise(pl.lit(f"{max_explicit+1}+"))
              .alias("complaints_bucket")
        )
        .group_by("complaints_bucket")
        .agg(pl.len().alias("n_petitioners"))
        .with_columns([
            (pl.col("n_petitioners") / total_petitioners * 100).round(2).alias("share_pct"),
        ])
    )
    freq_pd = freq.to_pandas()
    def bucket_sort_key(bucket):
        try:
            return int(bucket)
        except Exception:
            return 9999
    freq_pd["_sort"] = freq_pd["complaints_bucket"].apply(bucket_sort_key)
    freq_pd = freq_pd.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)
    freq_pd["cumulative_share_pct"] = freq_pd["share_pct"].cumsum().round(2)
    freq_pd.columns = ["No. of complaints", "No. of petitioners", "Share (%)", "Cumulative share (%)"]
    return freq_pd, total_petitioners

pet_counts = complaints_per_petitioner(df_clean)
freq_pd, total_petitioners = build_frequency_table(pet_counts)

print(f"Unique petitioners (excl. discarded): {total_petitioners:,}")
print(f"Summary stats:")
print(pet_counts.select("n_complaints").describe())

df_clean_july_2024_june_2025 = df_clean.filter(
    (pl.col("created_on") >= datetime(2024, 7, 1)) &
    (pl.col("created_on") < datetime(2025, 7, 1))
)
pet_counts_2425 = complaints_per_petitioner(df_clean_july_2024_june_2025)
freq_2425_pd, total_petitioners_2425 = build_frequency_table(pet_counts_2425)

print(f"2024-25 complaints (excl. discarded): {len(df_clean_july_2024_june_2025):,}")
print(f"2024-25 unique petitioners: {total_petitioners_2425:,}")
print(pet_counts_2425.select("n_complaints").describe())

In [ ]:
# ── Frequency distribution table ─────────────────────────────────────────────
display(freq_pd)

print("\n2024-25 petitioner frequency distribution:")
display(freq_2425_pd)

In [ ]:
# ── Save frequency table to Excel ────────────────────────────────────────────
out_path = OUTPUT / "tables" / "petitioner_frequency_distribution.xlsx"
out_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    freq_pd.to_excel(writer, sheet_name="frequency_table", index=False)
    freq_2425_pd.to_excel(writer, sheet_name="frequency_table_2024_25", index=False)
    
    # Summary stats sheets
    stats = pet_counts.select("n_complaints").describe().to_pandas()
    stats.to_excel(writer, sheet_name="summary_stats", index=False)
    stats_2425 = pet_counts_2425.select("n_complaints").describe().to_pandas()
    stats_2425.to_excel(writer, sheet_name="summary_stats_2024_25", index=False)

print(f"Saved: {out_path}")

---
## 2. Repeat-filer subcategory analysis

For petitioners who filed ≥2 complaints (excl. discarded): do they re-file the **same subcategory** (resolution failure / persistence) or **different subcategories** (active citizen with multiple distinct grievances)?

In [ ]:
# ── Repeat filers only ────────────────────────────────────────────────────────
repeat_filers = pet_counts.filter(pl.col("n_complaints") >= 2).select("petitioner_id")
print(f"Petitioners filing ≥2 complaints: {len(repeat_filers):,} ({len(repeat_filers)/total_petitioners*100:.1f}% of all petitioners)")

df_repeat = df_clean.join(repeat_filers, on="petitioner_id", how="inner")
print(f"Complaints from repeat filers: {len(df_repeat):,}")

In [ ]:
# ── Distinct subcategories per repeat-filing petitioner ───────────────────────
subcat_diversity = (
    df_repeat
    .group_by("petitioner_id")
    .agg([
        pl.len().alias("n_complaints"),
        pl.col("subcategory").n_unique().alias("n_distinct_subcategories"),
        pl.col("subcategory").first().alias("first_subcategory"),
    ])
    .with_columns(
        pl.when(pl.col("n_distinct_subcategories") == 1)
          .then(pl.lit("Same subcategory (re-filing)"))
          .otherwise(pl.lit("Different subcategories"))
          .alias("filing_pattern")
    )
)

pattern_summary = (
    subcat_diversity
    .group_by("filing_pattern")
    .agg(pl.len().alias("n_petitioners"))
    .with_columns(
        (pl.col("n_petitioners") / pl.col("n_petitioners").sum() * 100).round(1).alias("share_pct")
    )
    .sort("n_petitioners", descending=True)
)

print("Filing pattern among repeat filers:")
display(pattern_summary.to_pandas())

In [ ]:
# ── Among same-subcategory re-filers: what subcategory? ───────────────────────
same_subcat = subcat_diversity.filter(pl.col("filing_pattern") == "Same subcategory (re-filing)")

top_resubmit_subcats = (
    same_subcat
    .group_by("first_subcategory")
    .agg(pl.len().alias("n_petitioners"))
    .sort("n_petitioners", descending=True)
    .head(15)
)

print("Top subcategories among same-subcategory re-filers:")
display(top_resubmit_subcats.to_pandas())

In [ ]:
# ── By complaints-per-petitioner band: same vs different ─────────────────────
band_pattern = (
    subcat_diversity
    .with_columns(
        pl.when(pl.col("n_complaints") == 2).then(pl.lit("2"))
          .when(pl.col("n_complaints") <= 5).then(pl.lit("3–5"))
          .when(pl.col("n_complaints") <= 10).then(pl.lit("6–10"))
          .otherwise(pl.lit("11+"))
          .alias("complaint_band")
    )
    .group_by(["complaint_band", "filing_pattern"])
    .agg(pl.len().alias("n_petitioners"))
    .sort(["complaint_band", "filing_pattern"])
)

# Pivot for readability
band_pivot = band_pattern.to_pandas().pivot(
    index="complaint_band", columns="filing_pattern", values="n_petitioners"
).fillna(0).astype(int)
band_pivot["total"] = band_pivot.sum(axis=1)
band_pivot["same_pct"] = (band_pivot.get("Same subcategory (re-filing)", 0) / band_pivot["total"] * 100).round(1)

print("Filing pattern by complaint frequency band:")
display(band_pivot)

In [ ]:
# ── Save repeat-filer analysis ────────────────────────────────────────────────
out_path_subcat = OUTPUT / "tables" / "repeat_filer_subcategory_analysis.xlsx"

with pd.ExcelWriter(out_path_subcat, engine="openpyxl") as writer:
    pattern_summary.to_pandas().to_excel(writer, sheet_name="pattern_summary", index=False)
    top_resubmit_subcats.to_pandas().to_excel(writer, sheet_name="top_resubmit_subcats", index=False)
    band_pivot.to_excel(writer, sheet_name="by_complaint_band")

print(f"Saved: {out_path_subcat}")

---
## 3. Disposed-only resolution time histogram

Distribution of disposal time (days from `created_on` to `resolved_on`) for complaints with `status = 'Disposed'`, across all four fiscal years. This is the correct measure of genuine resolution speed — not the headline median which is pulled down by fast-closing discards.

In [ ]:
# ── Disposal time for disposed complaints ─────────────────────────────────────
df_disposed = (
    df
    .filter(pl.col("status") == "Disposed")
    .filter(pl.col("resolved_on").is_not_null())
    .with_columns(
        ((pl.col("resolved_on") - pl.col("created_on")).dt.total_days()).alias("disposal_days")
    )
    .filter(pl.col("disposal_days") >= 0)
    .filter(pl.col("fiscal_year").is_in(PRIMARY_FYS))
)

print(f"Disposed complaints with valid resolution date: {len(df_disposed):,}")
print(df_disposed.group_by("fiscal_year").agg([
    pl.len().alias("n"),
    pl.col("disposal_days").median().alias("median_days"),
    pl.col("disposal_days").mean().alias("mean_days"),
    pl.col("disposal_days").quantile(0.9).alias("p90_days"),
]).sort("fiscal_year"))

In [ ]:
# ── Histogram: disposal time by fiscal year ───────────────────────────────────
CAP_DAYS = 180

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
axes = axes.flatten()

for ax, fy in zip(axes, PRIMARY_FYS):
    data = (
        df_disposed
        .filter(pl.col('fiscal_year') == fy)
        .select('disposal_days')
        .to_pandas()['disposal_days']
        .clip(upper=CAP_DAYS)
    )
    median_d = data.median()
    n = len(data)

    ax.hist(data, bins=60, color=MAROON, alpha=0.88, edgecolor='white', linewidth=0.2)
    ax.axvline(median_d, color=NEAR_BLACK, linewidth=1.4, linestyle='--')

    # Place label left of line when median is near the cap, right otherwise
    near_cap = median_d > CAP_DAYS * 0.8
    x_label  = median_d - 4 if near_cap else median_d + 3
    ha_label = 'right'      if near_cap else 'left'
    ax.text(x_label, ax.get_ylim()[1] * 0.92,
            f'Median: {median_d:.0f}d', fontsize=8, color=NEAR_BLACK, ha=ha_label)

    ax.set_title(f'{fy}', fontsize=11)
    ax.set_xlabel('Days to disposal', fontsize=9, color='#555555')
    ax.set_ylabel('No. of complaints', fontsize=9, color='#555555')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.text(0.97, 0.97, f'n = {n:,}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='#777777')
    ax.spines['left'].set_color(GREY_AXIS)
    ax.spines['bottom'].set_color(GREY_AXIS)

fig.suptitle('Distribution of disposal time — disposed complaints only',
             fontsize=13, fontweight='bold', color=NEAR_BLACK, y=1.01)
fig.text(0.5, -0.02,
         f'Capped at {CAP_DAYS} days for display. Dashed line = median.',
         ha='center', fontsize=8, color='#777777', style='italic')

plt.tight_layout()
fig_path = OUTPUT / 'disposal_time_histogram.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()


In [ ]:
# ── Single overlaid histogram for presentation ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for fy, color in zip(PRIMARY_FYS, FY_COLORS):
    data = (
        df_disposed
        .filter(pl.col('fiscal_year') == fy)
        .select('disposal_days')
        .to_pandas()['disposal_days']
        .clip(upper=CAP_DAYS)
    )
    median_d = data.median()
    ax.hist(data, bins=60, alpha=0.60, color=color,
            label=f'{fy}  (median {median_d:.0f}d)', density=True)

ax.set_xlabel('Days to disposal', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Resolution time improving year on year (disposed complaints only)', fontsize=12)
ax.legend(fontsize=9, framealpha=0.9, edgecolor=GREY_GRID)
ax.spines['left'].set_color(GREY_AXIS)
ax.spines['bottom'].set_color(GREY_AXIS)
fig.text(0.5, -0.04,
         f'Capped at {CAP_DAYS} days. Density normalised so years are visually comparable.',
         ha='center', fontsize=8, color='#777777', style='italic')

plt.tight_layout()
fig_path2 = OUTPUT / 'disposal_time_overlaid.png'
plt.savefig(fig_path2, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path2}')
plt.show()


In [ ]:
# ── Median disposal time bar chart (July-June fiscal years) ──────────────────
df_bar_all = (
    df_disposed
    .with_columns(
        pl.when(pl.col('created_on').dt.month() >= 7)
          .then(pl.col('created_on').dt.year())
          .otherwise(pl.col('created_on').dt.year() - 1)
          .alias('fy_jul')
    )
    .filter(pl.col('fy_jul').is_in([2021, 2022, 2023, 2024]))
    .group_by('fy_jul')
    .agg(
        pl.col('disposal_days').median().alias('median_days'),
        pl.col('disposal_days').count().alias('n'),
    )
    .sort('fy_jul')
)

bar_df = df_bar_all.to_pandas()
fy_labels = {
    2021: '2021-22', 2022: '2022-23',
    2023: '2023-24', 2024: '2024-25',
}
bar_df['label'] = bar_df['fy_jul'].map(fy_labels)
bar_df = bar_df.sort_values('fy_jul')

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(bar_df['label'], bar_df['median_days'],
              color=MAROON, width=0.52, zorder=3)

for bar, val, n in zip(bars, bar_df['median_days'], bar_df['n']):
    # value label above bar
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{val:.0f}d', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=NEAR_BLACK)
    # sample size inside bar
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
            f'n={n:,}', ha='center', va='center',
            fontsize=8, color='white')

ax.set_ylabel('Median days to disposal', fontsize=10)
ax.set_ylim(0, bar_df['median_days'].max() * 1.28)
ax.spines['left'].set_color(GREY_AXIS)
ax.spines['bottom'].set_color(GREY_AXIS)
ax.set_title(
    'Median disposal time — disposed complaints only\n',
    fontsize=11,
)

plt.tight_layout()
fig.savefig(OUTPUT / 'disposal_time_median_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/disposal_time_median_bar.png')


---
## 4. Rural housing regression

Three OLS specifications estimated in **both levels and logs**:
- **M1**: month-year FE + office FE + 1{Online} + 1{Female}  (baseline)
- **M2**: M1 + district FE
- **M3**: M1 + block FE

M2 and M3 are separate regressions — block FE nests district FE, so they are not included jointly (collinear since districts are unions of blocks).

`office` encodes the first routing office (Collector, CM Office, Departments, Governor, Chief Secretary, SP) — 6 categories, 5 dummies, drop-first.

**Why log at all?** Disposal time is right-skewed; log-OLS gives better-behaved residuals and is conventional.  
**Why also levels?** Coefficients are directly interpretable as days — cleaner for policy communication.  
The R² comparison across specifications is the main goal: how much disposal-time variance do administrative variables capture vs what remains (attributable to complaint-level content: specificity, complexity, sentiment).


In [ ]:
# ── Rural housing disposed complaints ─────────────────────────────────────────
df_rh = (
    df_disposed
    .filter(pl.col("subcategory").is_in(RURAL_HOUSING_SUBCATS))
    .filter(pl.col("disposal_days") > 0)
    .filter(pl.col("district").is_not_null())
    .filter(pl.col("block").is_not_null())
    .filter(pl.col("mode").is_not_null())
    .filter(pl.col("petitioner_gender").is_not_null())
    .filter(pl.col("office").is_not_null())
    .filter(pl.col("fiscal_year").is_in(PRIMARY_FYS))
    .with_columns([
        pl.col("disposal_days").log1p().alias("log_disposal"),
        (pl.col("year").cast(pl.Utf8) + "-" + pl.col("month").cast(pl.Utf8)).alias("month_year"),
        # Collapse mode: Online = digital channels, Offline = in-person / paper
        pl.when(pl.col("mode").is_in([
            "Website", "Whatsapp", "Twitter", "Mobile", "Email", "Facebook"
        ])).then(pl.lit(1)).otherwise(pl.lit(0)).alias("online"),
        # Female indicator
        pl.when(pl.col("petitioner_gender").str.to_lowercase() == "female")
          .then(pl.lit(1)).otherwise(pl.lit(0)).alias("female"),
    ])
)

print(f"Rural housing disposed complaints for regression: {len(df_rh):,}")
print(df_rh.group_by("subcategory").len().sort("len", descending=True))
print("\nMode → online mapping:")
print(df_rh.group_by(["mode", "online"]).len().sort("len", descending=True))
print("\nOffice distribution:")
print(df_rh.group_by("office").len().sort("len", descending=True))


In [ ]:
# ── Convert to pandas for statsmodels ────────────────────────────────────────
reg_cols = ["log_disposal", "disposal_days", "district", "block", "office",
            "online", "female", "month_year", "fiscal_year", "subcategory"]
rh_pd = df_rh.select(reg_cols).to_pandas()

# Sanitise strings for dummy encoding (spaces/slashes/special chars → underscores)
for col in ["district", "block", "month_year", "office"]:
    rh_pd[col] = rh_pd[col].astype(str).str.strip().str.replace(r'[^\w]', '_', regex=True)

rh_pd = rh_pd.dropna(subset=["log_disposal", "disposal_days", "online", "female",
                               "district", "block", "month_year", "office"])
print(f"Regression sample: {len(rh_pd):,} rows")
print(f"Online share: {rh_pd['online'].mean():.1%}   Female share: {rh_pd['female'].mean():.1%}")
print(f"Office categories: {sorted(rh_pd['office'].unique())}")
rh_pd.head(3)


In [ ]:
import time

# ── Build design matrices once ────────────────────────────────────────────────
# Layout: [const | month-year FE | online | female | office FE | geo FE]
# online_idx and female_idx are fixed regardless of which geo FE is appended.

X_time   = pd.get_dummies(rh_pd["month_year"], drop_first=True, dtype=np.float32)
X_office = pd.get_dummies(rh_pd["office"],     drop_first=True, dtype=np.float32)
X_dist   = pd.get_dummies(rh_pd["district"],   drop_first=True, dtype=np.float32)
X_block  = pd.get_dummies(rh_pd["block"],      drop_first=True, dtype=np.float32)

scalars   = rh_pd[["online", "female"]].astype(np.float32).reset_index(drop=True)
intercept = pd.DataFrame({"const": np.ones(len(rh_pd), dtype=np.float32)})

# Indices for HC1 SEs — position is const + time dummies + online/female
online_idx = 1 + X_time.shape[1]       # intercept + month-year dummies
female_idx = online_idx + 1

X1 = pd.concat([intercept, X_time, scalars, X_office],          axis=1).values  # baseline
X2 = pd.concat([intercept, X_time, scalars, X_office, X_dist],  axis=1).values  # + district FE
X3 = pd.concat([intercept, X_time, scalars, X_office, X_block], axis=1).values  # + block FE

print(f"Office dummies: {X_office.shape[1]}  (categories: {list(X_office.columns)})")
print(f"Design matrix shapes: base={X1.shape}  dist={X2.shape}  block={X3.shape}")

# ── Fast OLS via numpy lstsq ──────────────────────────────────────────────────
def ols_numpy(X, y):
    """Returns (beta, R², adj-R², residuals) using numpy least-squares."""
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    resid  = y - X @ beta
    ss_res = resid @ resid
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = float(1 - ss_res / ss_tot)
    n, p = X.shape
    r2_adj = float(1 - (1 - r2) * (n - 1) / (n - p - 1))
    return beta, r2, r2_adj, resid

# ── Outcomes ──────────────────────────────────────────────────────────────────
y_log = rh_pd["log_disposal"].values.astype(np.float32)
y_lev = rh_pd["disposal_days"].values.astype(np.float32)

# ── Fit all 6 models ──────────────────────────────────────────────────────────
t0 = time.time()
results_raw = {}
for outcome_name, y in [("log", y_log), ("levels", y_lev)]:
    for spec_name, X in [("M1: baseline", X1), ("M2: + district FE", X2), ("M3: + block FE", X3)]:
        beta, r2, r2_adj, resid = ols_numpy(X, y)
        results_raw[(outcome_name, spec_name)] = {
            "beta": beta, "r2": r2, "r2_adj": r2_adj, "resid": resid, "X": X
        }
print(f"6 models solved in {time.time()-t0:.2f}s")

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for outcome_name, y in [("log(days+1)", y_log), ("days (levels)", y_lev)]:
    ok = "log" if "log" in outcome_name else "levels"
    for spec_name, X in [("M1: baseline", X1), ("M2: + district FE", X2), ("M3: + block FE", X3)]:
        r  = results_raw[(ok, spec_name)]
        b_on = float(r["beta"][online_idx])
        b_fe = float(r["beta"][female_idx])
        rows.append({
            "Model":     spec_name,
            "Outcome":   outcome_name,
            "N":         len(y),
            "R²":        round(r["r2"],    4),
            "Adj. R²":   round(r["r2_adj"],4),
            "β(online)": round(b_on, 3),
            "β(female)": round(b_fe, 3),
        })

results_df = pd.DataFrame(rows)
results_df["ΔR² vs M1"] = results_df.groupby("Outcome")["R²"].transform(
    lambda x: (x - x.iloc[0]).round(4)
)
display(results_df)

# ── HC1 SEs for online and female in M3 only ─────────────────────────────────
def hc1_se_for_indices(X, resid, idx_list):
    """HC1 sandwich SEs for specific coefficient indices only."""
    n, p = X.shape
    scale = n / (n - p)
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = (X * (resid[:, None] ** 2 * scale)).T @ X
    sandwich = XtX_inv @ meat @ XtX_inv
    return {i: float(np.sqrt(sandwich[i, i])) for i in idx_list}

t1 = time.time()
se_log_m3 = hc1_se_for_indices(X3, results_raw[("log",    "M3: + block FE")]["resid"], [online_idx, female_idx])
se_lev_m3 = hc1_se_for_indices(X3, results_raw[("levels", "M3: + block FE")]["resid"], [online_idx, female_idx])
print(f"HC1 SEs in {time.time()-t1:.2f}s")

print("\nM3 reported coefficients (HC1 SEs):")
for label, betas, ses in [
    ("log",    results_raw[("log",    "M3: + block FE")]["beta"], se_log_m3),
    ("levels", results_raw[("levels", "M3: + block FE")]["beta"], se_lev_m3),
]:
    unit = "" if label == "log" else "d"
    for name, idx in [("online", online_idx), ("female", female_idx)]:
        print(f"  [{label}] β({name}) = {betas[idx]:+.3f}{unit}  SE = {ses[idx]:.3f}{unit}")


In [ ]:
# ── Block-level coefficients from numpy M3 ────────────────────────────────────
# Block dummies start after: intercept(1) + month_year + online + female
block_start = 1 + X_time.shape[1] + 2  # = female_idx + 1

block_names_log   = X_block.columns.tolist()   # drop_first removed the reference block
block_names_level = X_block.columns.tolist()

beta_log_m3   = results_raw[("log",    "M3: + block FE")]["beta"]
beta_lev_m3   = results_raw[("levels", "M3: + block FE")]["beta"]

block_coefs = pd.DataFrame({
    "block":          block_names_log,
    "coef_log":       beta_log_m3[block_start : block_start + len(block_names_log)],
    "coef_days":      beta_lev_m3[block_start : block_start + len(block_names_level)],
})
block_coefs["implied_pct_change"] = (np.expm1(block_coefs["coef_log"]) * 100).round(1)
block_coefs = block_coefs.sort_values("coef_days", ascending=False).reset_index(drop=True)

print(f"Block coefficient range:")
print(f"  Levels: {block_coefs['coef_days'].min():.1f} to {block_coefs['coef_days'].max():.1f} days")
print(f"  Log:    {block_coefs['coef_log'].min():.3f} to {block_coefs['coef_log'].max():.3f}")
print(f"\nTop 10 slowest blocks (days above reference block):")
display(block_coefs.head(10))
print(f"\nTop 10 fastest blocks:")
display(block_coefs.tail(10))

In [ ]:
# ── Save regression results ───────────────────────────────────────────────────
reg_out = OUTPUT / "tables" / "rural_housing_regression.xlsx"

with pd.ExcelWriter(reg_out, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="model_comparison", index=False)
    block_coefs.to_excel(writer, sheet_name="block_coefficients", index=False)

    # SE table for the two reported coefficients
    se_table = pd.DataFrame([
        {"outcome": "log(days+1)", "variable": "online", "beta": float(beta_log_m3[online_idx]), "SE_HC1": se_log_m3[online_idx]},
        {"outcome": "log(days+1)", "variable": "female", "beta": float(beta_log_m3[female_idx]), "SE_HC1": se_log_m3[female_idx]},
        {"outcome": "days (levels)", "variable": "online", "beta": float(beta_lev_m3[online_idx]), "SE_HC1": se_lev_m3[online_idx]},
        {"outcome": "days (levels)", "variable": "female", "beta": float(beta_lev_m3[female_idx]), "SE_HC1": se_lev_m3[female_idx]},
    ])
    se_table.to_excel(writer, sheet_name="reported_coef_SEs", index=False)

print(f"Saved: {reg_out}")


---
## 5. Resolution figures

Extend the existing presentation notebook with publication-ready figures for the resolution section. This block reuses the status-grouping logic from `status.ipynb`, the regression outputs already computed above, and the PMAY routing logic from `actionhistory.ipynb`.


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe
import re
import sqlite3
import textwrap
from IPython.display import Markdown

RESOLUTION_YEARS = ["2023-2024", "2024-2025"]
RESOLUTION_YEAR_DISPLAY = {
    "2023-2024": "2023-24",
    "2024-2025": "2024-25",
}
RESOLUTION_YEAR_SHORT = {
    "2023-2024": "2023-24",
    "2024-2025": "2024-25",
}
PMAY_YEAR = "2024-2025"
PMAY_NAME = "IAY/MKY/BPGY/PMAY"
ONLINE_MODES = {"Website", "Whatsapp", "Twitter", "Mobile", "Email", "Facebook"}

DISPOSED_WITH_BENEFIT = "Disposed with benefit"
DISPOSED_WITHOUT_BENEFIT = "Disposed"
STATUS_ORDER = ["Discard", DISPOSED_WITHOUT_BENEFIT, DISPOSED_WITH_BENEFIT, "Open"]
STATUS_COLORS = {
    "Discard": "#C65A5A",
    DISPOSED_WITHOUT_BENEFIT: MAROON,
    DISPOSED_WITH_BENEFIT: "#C48A00",
    "Open": "#B8BDC7",
}
GOOD_GREEN = "#2C7A4B"
BAD_RED = "#A33D3D"
MID_GREY = "#6D737A"
LIGHT_FILL = "#F7F4EF"

RESOLUTION_FILES = {
    "status_shift": OUTPUT / "resolution_status_shift.png",
    "status_shift_pie": OUTPUT / "resolution_status_shift_pie.png",
    "scorecard": OUTPUT / "resolution_scorecard.png",
    "regression": OUTPUT / "regression_block_vs_district.png",
    "open_monthly": OUTPUT / "resolution_open_monthly_trend.png",
    "routing_compare": OUTPUT / "routing_route2_vs_route4.png",
    "routing_steps": OUTPUT / "routing_step_decomposition.png",
    "routing_upper_bound": OUTPUT / "routing_counterfactual_upper_bound.png",
}

for pth in RESOLUTION_FILES.values():
    pth.parent.mkdir(parents=True, exist_ok=True)


def save_polished(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0.16, facecolor="white")
    plt.close(fig)
    print(f"Saved: {path}")


def july_june_year_label(series):
    years = series.dt.year
    months = series.dt.month
    return np.where(
        months >= 7,
        years.astype(str) + "-" + (years + 1).astype(str),
        (years - 1).astype(str) + "-" + years.astype(str),
    )


def publication_status(status, benefitted):
    status = "" if status is None else str(status).strip()
    benefitted = "No" if benefitted is None else str(benefitted).strip()
    if status == "Discard":
        return "Discard"
    if status == "Disposed" and benefitted.lower() == "yes":
        return DISPOSED_WITH_BENEFIT
    if status == "Disposed":
        return DISPOSED_WITHOUT_BENEFIT
    return "Open"


def friendly_authority(authority):
    mapping = {"CM": "CM Grievance Cell", "COLLECTOR": "Collector", "BDO": "BDO"}
    return mapping.get(authority, authority.title())


def direct_label(ax, x, y, text, color="white", size=10, weight="bold"):
    ax.text(
        x, y, text,
        ha="center", va="center", fontsize=size, color=color, fontweight=weight,
        path_effects=[pe.withStroke(linewidth=2, foreground="white", alpha=0.18)] if color != "white" else None,
    )


def draw_card(
    ax, xy, width, height, title, value, subtitle, facecolor, accent, value_color=NEAR_BLACK,
    title_size=11, value_size=22, subtitle_size=10, wrap_chars=None,
):
    x, y = xy
    card = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.012,rounding_size=0.03",
        linewidth=0,
        facecolor=facecolor,
        transform=ax.transAxes,
        zorder=1,
    )
    ax.add_patch(card)
    accent_band = FancyBboxPatch(
        (x, y + height - 0.03), width, 0.03,
        boxstyle="round,pad=0.0,rounding_size=0.03",
        linewidth=0,
        facecolor=accent,
        transform=ax.transAxes,
        zorder=2,
    )
    ax.add_patch(accent_band)
    subtitle_wrapped = textwrap.fill(subtitle, width=wrap_chars or max(28, int(width * 85)))
    compact = height < 0.25
    title_y = y + height - (0.045 if compact else 0.07)
    value_y = y + height * (0.42 if compact else 0.47)
    subtitle_y = y + (0.018 if compact else 0.04)
    ax.text(x + 0.03, title_y, title, transform=ax.transAxes,
            ha="left", va="top", fontsize=title_size, color=MID_GREY, fontweight="bold")
    ax.text(x + 0.03, value_y, value, transform=ax.transAxes,
            ha="left", va="center", fontsize=value_size, color=value_color, fontweight="bold")
    if subtitle_wrapped.strip():
        ax.text(x + 0.03, subtitle_y, subtitle_wrapped, transform=ax.transAxes,
                ha="left", va="bottom", fontsize=subtitle_size, color=NEAR_BLACK, linespacing=1.18)


def draw_route_box(ax, x, y, w, h, label, facecolor, edgecolor=GREY_AXIS, text_color=NEAR_BLACK):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.008,rounding_size=0.02",
        linewidth=1.0,
        edgecolor=edgecolor,
        facecolor=facecolor,
        transform=ax.transAxes,
    )
    ax.add_patch(box)
    ax.text(x + w / 2, y + h / 2, label, transform=ax.transAxes,
            ha="center", va="center", fontsize=10, color=text_color, fontweight="bold")


def draw_route_arrow(ax, x1, x2, y):
    arrow = FancyArrowPatch(
        (x1, y), (x2, y),
        transform=ax.transAxes,
        arrowstyle="-|>",
        mutation_scale=12,
        linewidth=1.3,
        color=GREY_AXIS,
    )
    ax.add_patch(arrow)

print('Resolution figure helpers ready.')


In [ ]:
# ── Status composition and scorecard inputs (July-June year) ─────────────────
status_pd = (
    df
    .select(["ticket_no", "status", "benefitted", "created_on", "resolved_on"])
    .to_pandas()
)
status_pd["created_on"] = pd.to_datetime(status_pd["created_on"], errors="coerce")
status_pd["resolved_on"] = pd.to_datetime(status_pd["resolved_on"], errors="coerce")
status_pd = status_pd[status_pd["created_on"].notna()].copy()
status_pd["custom_year"] = july_june_year_label(status_pd["created_on"])
status_pd = status_pd[status_pd["custom_year"].isin(RESOLUTION_YEARS)].copy()
status_pd["status"] = status_pd["status"].fillna("Unknown").astype(str).str.strip()
status_pd["benefitted"] = status_pd["benefitted"].fillna("No").astype(str).str.strip()
status_pd["status_group"] = [publication_status(s, b) for s, b in zip(status_pd["status"], status_pd["benefitted"])]

status_counts = (
    status_pd
    .groupby(["custom_year", "status_group"], as_index=False)["ticket_no"]
    .nunique()
    .rename(columns={"ticket_no": "count"})
)
year_totals = status_counts.groupby("custom_year", as_index=False)["count"].sum().rename(columns={"count": "total_cases"})
status_counts = status_counts.merge(year_totals, on="custom_year", how="left")
status_counts["percentage"] = (status_counts["count"] / status_counts["total_cases"] * 100).round(2)
status_counts["status_group"] = pd.Categorical(status_counts["status_group"], STATUS_ORDER, ordered=True)
status_counts = status_counts.sort_values(["custom_year", "status_group"]).reset_index(drop=True)

resolved_status = status_pd[status_pd["resolved_on"].notna()].copy()
resolved_status["resolution_days"] = (
    (resolved_status["resolved_on"] - resolved_status["created_on"]).dt.total_seconds() / (24 * 3600)
)
resolved_status = resolved_status[resolved_status["resolution_days"] >= 0].copy()

yearly_status_exact = (
    resolved_status
    .groupby(["custom_year", "status"], as_index=False)
    .agg(
        count=("ticket_no", "nunique"),
        mean_days=("resolution_days", "mean"),
        median_days=("resolution_days", "median"),
    )
)
yearly_status_exact[["mean_days", "median_days"]] = yearly_status_exact[["mean_days", "median_days"]].round(2)


def metric_value(frame, year_label, group_label, column, key="status_group"):
    row = frame[(frame["custom_year"] == year_label) & (frame[key] == group_label)]
    if row.empty:
        return float('nan')
    return float(row.iloc[0][column])


scorecard_metrics = {
    "disposed_median_2324": metric_value(yearly_status_exact, "2023-2024", "Disposed", "median_days", key="status"),
    "disposed_median_2425": metric_value(yearly_status_exact, "2024-2025", "Disposed", "median_days", key="status"),
    "discard_median_2324": metric_value(yearly_status_exact, "2023-2024", "Discard", "median_days", key="status"),
    "discard_median_2425": metric_value(yearly_status_exact, "2024-2025", "Discard", "median_days", key="status"),
    "benefit_pct_2324": metric_value(status_counts, "2023-2024", DISPOSED_WITH_BENEFIT, "percentage"),
    "benefit_pct_2425": metric_value(status_counts, "2024-2025", DISPOSED_WITH_BENEFIT, "percentage"),
    "open_count_2324": metric_value(status_counts, "2023-2024", "Open", "count"),
    "open_count_2425": metric_value(status_counts, "2024-2025", "Open", "count"),
}

status_shift = (
    status_counts
    .pivot(index="custom_year", columns="status_group", values="percentage")
    .reindex(RESOLUTION_YEARS)
    .reindex(columns=STATUS_ORDER)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(10.2, 7.0))
fig.subplots_adjust(left=0.10, right=0.92, top=0.96, bottom=0.18)
positions = np.arange(len(RESOLUTION_YEARS))
width = 0.6
bottom = np.zeros(len(RESOLUTION_YEARS))
segment_centers = {}

for status_group in STATUS_ORDER:
    values = status_shift[status_group].values
    bars = ax.bar(
        positions, values, width=width, bottom=bottom,
        color=STATUS_COLORS[status_group], edgecolor="white", linewidth=1.2,
        label=status_group,
    )
    for i, (bar, val) in enumerate(zip(bars, values)):
        center_y = bottom[i] + val / 2
        segment_centers[(RESOLUTION_YEARS[i], status_group)] = center_y
        if val >= 3.8:
            text_color = "white" if status_group != DISPOSED_WITH_BENEFIT else NEAR_BLACK
            direct_label(ax, bar.get_x() + bar.get_width() / 2, center_y, f"{val:.1f}%", color=text_color, size=10)
    bottom += values

ax.set_ylim(0, 100)
ax.set_xticks(positions, [RESOLUTION_YEAR_DISPLAY[y] for y in RESOLUTION_YEARS])
ax.set_yticks(np.arange(0, 101, 20))
ax.set_ylabel("Share of all complaints (%)")
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.spines["left"].set_color(GREY_AXIS)
ax.spines["bottom"].set_color(GREY_AXIS)
legend = ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
for txt in legend.get_texts():
    txt.set_color(NEAR_BLACK)

ax.annotate(
    f"Benefitted share fell\n{scorecard_metrics['benefit_pct_2324']:.1f}% → {scorecard_metrics['benefit_pct_2425']:.1f}%",
    xy=(positions[1], segment_centers[("2024-2025", DISPOSED_WITH_BENEFIT)]),
    xytext=(1.28, 54),
    textcoords="data",
    arrowprops=dict(arrowstyle="-|>", color=STATUS_COLORS[DISPOSED_WITH_BENEFIT], lw=1.5),
    fontsize=10,
    color=NEAR_BLACK,
    ha="left",
    va="center",
    bbox=dict(boxstyle="round,pad=0.35", fc="#FFF5D8", ec="none"),
)

save_polished(fig, RESOLUTION_FILES["status_shift"])

fig, ax = plt.subplots(figsize=(11.6, 6.2))
ax.axis("off")
ax.text(0.03, 0.90, "What improved", fontsize=12, fontweight="bold", color=GOOD_GREEN, transform=ax.transAxes)
ax.text(0.54, 0.90, "What worsened or the headline hides", fontsize=12, fontweight="bold", color=BAD_RED, transform=ax.transAxes)

draw_card(
    ax, (0.03, 0.12), 0.43, 0.68,
    "Disposed-only median", f"{scorecard_metrics['disposed_median_2324']:.0f}d → {scorecard_metrics['disposed_median_2425']:.0f}d",
    "Median disposal time for exact status = Disposed.",
    facecolor="#E9F5EE", accent=GOOD_GREEN, value_color=GOOD_GREEN,
)

draw_card(
    ax, (0.54, 0.62), 0.41, 0.18,
    "Median discard time", f"{scorecard_metrics['discard_median_2324']:.0f}d → {scorecard_metrics['discard_median_2425']:.0f}d",
    "",
    facecolor="#F8ECEC", accent=BAD_RED, value_size=20, subtitle_size=8.3, wrap_chars=38, title_size=10.5,
)

draw_card(
    ax, (0.54, 0.39), 0.41, 0.17,
    "Disposed with benefit", f"{scorecard_metrics['benefit_pct_2324']:.2f}% → {scorecard_metrics['benefit_pct_2425']:.2f}%",
    "",
    facecolor="#FFF4E0", accent=STATUS_COLORS[DISPOSED_WITH_BENEFIT], value_color=STATUS_COLORS[DISPOSED_WITH_BENEFIT], value_size=20, subtitle_size=8.3, wrap_chars=36, title_size=10.5,
)

draw_card(
    ax, (0.54, 0.14), 0.41, 0.19,
    "Open cases", f"{int(scorecard_metrics['open_count_2324']):,} → {int(scorecard_metrics['open_count_2425']):,}",
    "",
    facecolor="#EEF1F5", accent=STATUS_COLORS["Open"], value_size=20, subtitle_size=8.3, wrap_chars=36, title_size=10.5,
)

open_monthly = (
    status_pd.loc[status_pd["status_group"] == "Open", ["ticket_no", "created_on"]]
    .assign(month=lambda x: x["created_on"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)["ticket_no"]
    .nunique()
    .rename(columns={"ticket_no": "open_count"})
)
month_range = pd.date_range(
    status_pd["created_on"].min().to_period("M").to_timestamp(),
    status_pd["created_on"].max().to_period("M").to_timestamp(),
    freq="MS",
)
open_monthly = (
    open_monthly
    .set_index("month")
    .reindex(month_range, fill_value=0)
    .rename_axis("month")
    .reset_index()
)
peak_row = open_monthly.loc[open_monthly["open_count"].idxmax()]
latest_row = open_monthly.iloc[-1]

save_polished(fig, RESOLUTION_FILES["scorecard"])

fig2, ax2 = plt.subplots(figsize=(10.4, 4.9))
ax2.plot(open_monthly["month"], open_monthly["open_count"], color=MAROON, lw=2.6, solid_capstyle="round")
ax2.fill_between(open_monthly["month"], open_monthly["open_count"], color=MAROON, alpha=0.08)
ax2.scatter(peak_row["month"], peak_row["open_count"], color=MAROON, s=42, zorder=4)
ax2.scatter(latest_row["month"], latest_row["open_count"], color=MAROON, s=42, zorder=4)
ax2.annotate(
    f"Peak {int(peak_row['open_count']):,}",
    xy=(peak_row["month"], peak_row["open_count"]),
    xytext=(peak_row["month"], peak_row["open_count"] + open_monthly["open_count"].max() * 0.10),
    textcoords="data",
    ha="center",
    fontsize=9.5,
    color=NEAR_BLACK,
    arrowprops=dict(arrowstyle="-|>", color=MAROON_MID, lw=1.1),
    bbox=dict(boxstyle="round,pad=0.22", fc="#F7F4EF", ec="none"),
)
ax2.text(
    latest_row["month"],
    latest_row["open_count"] + open_monthly["open_count"].max() * 0.06,
    f"Latest {int(latest_row['open_count']):,}",
    fontsize=9.5,
    color=NEAR_BLACK,
    ha="right",
)
tick_months = open_monthly.loc[open_monthly["month"].dt.month.isin([1, 4, 7, 10]), "month"]
ax2.set_xticks(tick_months)
ax2.set_xticklabels([dt.strftime("%b %Y") for dt in tick_months], rotation=0)
ax2.set_ylabel("Open complaints created")
ax2.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax2.grid(axis="y", linestyle="--", alpha=0.25)
ax2.spines["left"].set_color(GREY_AXIS)
ax2.spines["bottom"].set_color(GREY_AXIS)
ax2.margins(x=0.02)

save_polished(fig2, RESOLUTION_FILES["open_monthly"])

display(status_counts)
display(yearly_status_exact)
print('Scorecard metrics:', scorecard_metrics)


In [ ]:
# ── Status composition pie variant ───────────────────────────────────────────
pie_label_radius = {
    "Discard": 0.78,
    DISPOSED_WITHOUT_BENEFIT: 0.58,
    DISPOSED_WITH_BENEFIT: 0.78,
    "Open": 0.80,
}
benefit_anchors = {}

fig, axes = plt.subplots(1, 2, figsize=(11.4, 6.0), subplot_kw=dict(aspect="equal"))
fig.subplots_adjust(left=0.05, right=0.95, top=0.90, bottom=0.18, wspace=0.04)

for ax, year in zip(axes, RESOLUTION_YEARS):
    values = status_shift.loc[year, STATUS_ORDER].values
    wedges, _ = ax.pie(
        values,
        colors=[STATUS_COLORS[s] for s in STATUS_ORDER],
        startangle=100,
        counterclock=False,
        wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2),
    )
    ax.text(0, 0.10, RESOLUTION_YEAR_DISPLAY[year], ha="center", va="center",
            fontsize=15, fontweight="bold", color=NEAR_BLACK)
    total_cases = int(year_totals.loc[year_totals["custom_year"] == year, "total_cases"].iloc[0])
    ax.text(0, -0.10, f"{total_cases:,}\ncases", ha="center", va="center",
            fontsize=10, color=MID_GREY, linespacing=1.15)

    for wedge, status_group, val in zip(wedges, STATUS_ORDER, values):
        theta = np.deg2rad((wedge.theta1 + wedge.theta2) / 2)
        if status_group == DISPOSED_WITH_BENEFIT:
            benefit_anchors[year] = (np.cos(theta) * 0.88, np.sin(theta) * 0.88)
        if val < 4.0:
            continue
        x = np.cos(theta) * pie_label_radius[status_group]
        y = np.sin(theta) * pie_label_radius[status_group]
        text_color = "white" if status_group in {"Discard", DISPOSED_WITHOUT_BENEFIT} else NEAR_BLACK
        direct_label(ax, x, y, f"{val:.1f}%", color=text_color, size=10)

    ax.set_xticks([])
    ax.set_yticks([])

axes[1].annotate(
    f"Benefitted share fell\n{scorecard_metrics['benefit_pct_2324']:.1f}% → {scorecard_metrics['benefit_pct_2425']:.1f}%",
    xy=benefit_anchors["2024-2025"],
    xytext=(0.98, 0.90),
    textcoords="axes fraction",
    arrowprops=dict(arrowstyle="-|>", color=STATUS_COLORS[DISPOSED_WITH_BENEFIT], lw=1.5),
    fontsize=10,
    color=NEAR_BLACK,
    ha="left",
    va="top",
    bbox=dict(boxstyle="round,pad=0.35", fc="#FFF5D8", ec="none"),
)

legend = fig.legend(wedges, STATUS_ORDER, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 0.03), frameon=False)
for txt in legend.get_texts():
    txt.set_color(NEAR_BLACK)

fig.text(
    0.50,
    0.12,
    "Discard and open shares widened in 2024-25 while the benefitted slice narrowed.",
    ha="center",
    fontsize=10,
    color=MID_GREY,
)

save_polished(fig, RESOLUTION_FILES["status_shift_pie"])


In [ ]:

# ── Regression takeaway figure ───────────────────────────────────────────────
log_models = results_df[results_df["Outcome"] == "log(days+1)"].copy().reset_index(drop=True)
log_models["label"] = ["M1", "M1 +\nDistrict effects", "M1 +\nBlock effects"]
log_models["delta_baseline"] = (log_models["R²"] - log_models.loc[0, "R²"]).round(4)
block_vs_district_ratio = log_models.loc[2, "delta_baseline"] / log_models.loc[1, "delta_baseline"]

fig, ax = plt.subplots(figsize=(8.6, 5.6))
bar_colors = ["#D9C7C7", MAROON_MID, MAROON]
bars = ax.bar(log_models["label"], log_models["R²"], color=bar_colors, width=0.58, zorder=3)
ax.set_ylim(0, 0.43)
ax.set_ylabel("R²")
ax.grid(axis="y", linestyle="--", alpha=0.25, zorder=0)
ax.spines["left"].set_color(GREY_AXIS)
ax.spines["bottom"].set_color(GREY_AXIS)

for bar, r2 in zip(bars, log_models["R²"]):
    ax.text(bar.get_x() + bar.get_width()/2, r2 + 0.012, f"R² = {r2:.3f}", ha="center", va="bottom",
            fontsize=10, color=NEAR_BLACK, fontweight="bold")

ax.annotate(
    f"District adds +{log_models.loc[1, 'delta_baseline']:.3f}",
    xy=(1, log_models.loc[1, "R²"]), xytext=(0.55, 0.34),
    textcoords="data",
    arrowprops=dict(arrowstyle="-|>", color=MAROON_MID, lw=1.4),
    fontsize=10, color=NEAR_BLACK,
    bbox=dict(boxstyle="round,pad=0.25", fc="#F6ECEC", ec="none"),
)
ax.annotate(
    f"Block adds +{log_models.loc[2, 'delta_baseline']:.3f}\n({block_vs_district_ratio:.1f}× district)",
    xy=(2, log_models.loc[2, "R²"]), xytext=(1.18, 0.305),
    textcoords="data",
    arrowprops=dict(arrowstyle="-|>", color=MAROON, lw=1.4),
    fontsize=10, color=NEAR_BLACK,
    bbox=dict(boxstyle="round,pad=0.25", fc="#F3E3E3", ec="none"),
)

save_polished(fig, RESOLUTION_FILES["regression"])
print(log_models[["Model", "R²", "delta_baseline"]])


In [ ]:
# ── PMAY routing metrics from action_history (July-June year) ────────────────
pmay_routes = (
    df
    .with_columns([
        pl.when(pl.col("created_on").dt.month() >= 7)
          .then(pl.col("created_on").dt.year().cast(pl.Utf8) + "-" + (pl.col("created_on").dt.year() + 1).cast(pl.Utf8))
          .otherwise((pl.col("created_on").dt.year() - 1).cast(pl.Utf8) + "-" + pl.col("created_on").dt.year().cast(pl.Utf8))
          .alias("custom_year"),
    ])
    .filter(
        (pl.col("custom_year") == PMAY_YEAR) &
        (pl.col("subcategory") == PMAY_NAME) &
        (pl.col("status") == "Disposed") &
        (pl.col("benefitted").fill_null("No").str.to_lowercase() == pl.lit("no")) &
        pl.col("resolved_on").is_not_null()
    )
    .with_columns([
        ((pl.col("resolved_on") - pl.col("created_on")).dt.total_days()).alias("resolution_days"),
        pl.when(pl.col("mode").is_in(list(ONLINE_MODES)))
          .then(pl.lit("Online"))
          .otherwise(pl.lit("Offline"))
          .alias("mode_group"),
    ])
    .filter(pl.col("resolution_days") >= 0)
    .select(["ticket_no", "created_on", "resolved_on", "resolution_days", "mode_group"])
    .to_pandas()
)

route_query = """
SELECT
    a.id,
    a.ticket_no,
    a.action_taken_date,
    a.action_taken_by,
    a.complaint_status_with_authority
FROM action_history a
INNER JOIN complaints c
    ON a.ticket_no = c.ticket_no
WHERE c.subcategory = ?
  AND c.status = 'Disposed'
  AND lower(coalesce(c.benefitted, '')) = 'no'
  AND c.created_on >= '2024-07-01'
  AND c.created_on < '2025-07-01'
ORDER BY a.ticket_no, a.action_taken_date, a.id
"""

with sqlite3.connect(str(directories.RAW_DATA / 'grievance.db')) as conn:
    ah_pmay = pd.read_sql_query(route_query, conn, params=[PMAY_NAME], parse_dates=["action_taken_date"])


def authority_from_status(row):
    status = "" if pd.isna(row["complaint_status_with_authority"]) else str(row["complaint_status_with_authority"]).strip()
    taken_by = "" if pd.isna(row["action_taken_by"]) else str(row["action_taken_by"]).strip()
    if " - " in status:
        return status.split(" - ", 1)[1].strip()
    return taken_by


def clean_authority(value):
    s = "" if pd.isna(value) else str(value).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*,\s*", ", ", s)
    if "," in s:
        s = s.split(",", 1)[0].strip()
    s = s.upper()
    s = s.replace("COLLECTOR & DM", "COLLECTOR")
    s = s.replace("COLLECTOR AND DM", "COLLECTOR")
    s = s.replace("BLOCK DEVELOPMENT OFFICER", "BDO")
    s = s.replace("B.D.O", "BDO")
    s = s.replace("CHIEF MINISTER", "CM")
    s = s.replace("CMO", "CM")
    if "CM GRIEVANCE CELL" in s:
        s = "CM"
    s = re.sub(r"\s{2,}", " ", s).strip(" -_/,.")
    return s

ah_pmay["authority_raw"] = ah_pmay.apply(authority_from_status, axis=1)
ah_pmay["authority_clean"] = ah_pmay["authority_raw"].apply(clean_authority)
ah_pmay = ah_pmay[
    ah_pmay["action_taken_date"].notna() &
    ah_pmay["authority_clean"].ne("")
].copy()

records = []
current_ticket = None
prev_auth = None
step_no = 0
for ticket_no, action_taken_date, authority_clean in ah_pmay[["ticket_no", "action_taken_date", "authority_clean"]].itertuples(index=False):
    if ticket_no != current_ticket:
        current_ticket = ticket_no
        prev_auth = None
        step_no = 0
    if authority_clean != prev_auth:
        step_no += 1
        records.append((ticket_no, step_no, authority_clean, action_taken_date))
        prev_auth = authority_clean

step_rows = pd.DataFrame(records, columns=["ticket_no", "step_no", "authority_clean", "step_start"])
route_lists = step_rows.groupby("ticket_no")["authority_clean"].apply(list).reset_index(name="route_list")
route_stats = route_lists.merge(pmay_routes[["ticket_no", "resolution_days", "mode_group"]], on="ticket_no", how="inner")
route_stats["route_length"] = route_stats["route_list"].apply(len)
route_stats["route_string"] = route_stats["route_list"].apply(lambda xs: " → ".join(friendly_authority(x) for x in xs))

route_length_summary = (
    route_stats[route_stats["route_length"].isin([2, 4])]
    .groupby("route_length", as_index=False)
    .agg(
        tickets=("ticket_no", "count"),
        mean_days=("resolution_days", "mean"),
        median_days=("resolution_days", "median"),
    )
)
route_length_summary[["mean_days", "median_days"]] = route_length_summary[["mean_days", "median_days"]].round(2)
route_length_summary["share_pct"] = (route_length_summary["tickets"] / len(route_stats) * 100).round(2)

route4_tickets = route_stats[route_stats["route_length"] == 4].copy()
step_name_wide = step_rows.pivot(index="ticket_no", columns="step_no", values="authority_clean").rename(columns=lambda c: f"step_{c}_name")
step_start_wide = step_rows.pivot(index="ticket_no", columns="step_no", values="step_start").rename(columns=lambda c: f"step_{c}_start")
route4_detail = route4_tickets.merge(step_name_wide, on="ticket_no", how="left").merge(step_start_wide, on="ticket_no", how="left")
route4_detail = route4_detail.merge(pmay_routes[["ticket_no", "created_on", "resolved_on"]], on="ticket_no", how="left")

for left, right, name in [(1, 2, "gap_1_2_days"), (2, 3, "gap_2_3_days"), (3, 4, "gap_3_4_days")]:
    route4_detail[name] = (
        (route4_detail[f"step_{right}_start"] - route4_detail[f"step_{left}_start"]).dt.total_seconds() / (24 * 3600)
    )
route4_detail["gap_4_resolve_days"] = (
    (route4_detail["resolved_on"] - route4_detail["step_4_start"]).dt.total_seconds() / (24 * 3600)
)
route4_detail["created_to_step1_days"] = (
    (route4_detail["step_1_start"] - route4_detail["created_on"]).dt.total_seconds() / (24 * 3600)
)
route4_detail = route4_detail[route4_detail["resolution_days"].notna() & (route4_detail["resolution_days"] >= 0)].copy()

target_route4 = route4_detail[
    route4_detail["route_string"] == "CM Grievance Cell → Collector → BDO → Collector"
].copy()
route4_pattern_share = round(len(target_route4) / len(route4_tickets) * 100, 2)

step_breakup = pd.DataFrame({
    "step": [
        "CM Grievance Cell → Collector",
        "Collector → BDO",
        "BDO → Collector",
        "Total resolution",
    ],
    "mean_days": [
        target_route4["gap_1_2_days"].mean(),
        target_route4["gap_2_3_days"].mean(),
        target_route4["gap_3_4_days"].mean(),
        target_route4["resolution_days"].mean(),
    ],
    "median_days": [
        target_route4["gap_1_2_days"].median(),
        target_route4["gap_2_3_days"].median(),
        target_route4["gap_3_4_days"].median(),
        target_route4["resolution_days"].median(),
    ],
}).round(2)

route2_summary = route_length_summary.loc[route_length_summary["route_length"] == 2].iloc[0].to_dict()
route4_summary = route_length_summary.loc[route_length_summary["route_length"] == 4].iloc[0].to_dict()
route_counterfactual = {
    "current_median": float(route4_summary["median_days"]),
    "benchmark_median": float(route2_summary["median_days"]),
    "upper_bound_saved": float(route4_summary["median_days"] - route2_summary["median_days"]),
}

print(route_length_summary)
print("\nDominant route-4 pattern share within route length 4:", route4_pattern_share)
display(step_breakup)


In [ ]:
# ── Routing figures and notebook-side execution summary ─────────────────────
fig, ax = plt.subplots(figsize=(12.2, 5.8))
ax.axis("off")

lane_y = {2: 0.68, 4: 0.30}
box_w = 0.11
box_h = 0.11
route_layout = {
    2: [(0.21, "Collector"), (0.40, "BDO"), (0.59, "Close")],
    4: [(0.19, "CM Cell"), (0.36, "Collector"), (0.53, "BDO"), (0.70, "Collector return"), (0.87, "Close")],
}
route_fill = {2: "#E8F1ED", 4: "#F7E8E8"}
route_stats_map = {2: route2_summary, 4: route4_summary}

for route_len in [2, 4]:
    y = lane_y[route_len]
    ax.text(0.03, y + 0.05, f"Route {route_len}", fontsize=12, fontweight="bold", color=NEAR_BLACK, transform=ax.transAxes)
    ax.text(0.03, y - 0.04, f"{round(route_stats_map[route_len]['share_pct']):.0f}% of cases | median {route_stats_map[route_len]['median_days']:.0f}d",
            fontsize=10.0, color=MID_GREY, transform=ax.transAxes)
    positions = route_layout[route_len]
    for idx, (x, label) in enumerate(positions):
        draw_route_box(ax, x, y, box_w, box_h, label, route_fill[route_len])
        if idx < len(positions) - 1:
            draw_route_arrow(ax, x + box_w, positions[idx + 1][0], y + box_h / 2)

ax.text(0.64, 0.16,
        f"Within route-length 4, the dominant pattern\nCM Grievance Cell → Collector → BDO → Collector\ncovers {route4_pattern_share:.1f}% of cases.",
        fontsize=9.5, color=NEAR_BLACK, transform=ax.transAxes,
        bbox=dict(boxstyle="round,pad=0.35", fc="#F7F4EF", ec="none"))

save_polished(fig, RESOLUTION_FILES["routing_compare"])

plot_steps = step_breakup.iloc[:3].copy()
plot_steps["display"] = ["Step 1  CM Cell → Collector", "Step 2  Collector → BDO", "Step 3  BDO → Collector"]
plot_steps = plot_steps.iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10.0, 6.0))
fig.subplots_adjust(left=0.24, right=0.96, top=0.96, bottom=0.20)
y = np.arange(len(plot_steps))
ax.hlines(y, 0, plot_steps["median_days"], color=MAROON_MID, lw=9, alpha=0.85)
ax.scatter(plot_steps["median_days"], y, color=MAROON, s=80, zorder=3, label="Median")
ax.scatter(plot_steps["mean_days"], y, color=STATUS_COLORS[DISPOSED_WITH_BENEFIT], s=110, zorder=4, marker="o", label="Mean")
max_x = float(plot_steps[["mean_days", "median_days"]].max().max())
for yi, row in plot_steps.iterrows():
    ax.text(max_x + 2.4, yi, f"median {row['median_days']:.0f}d | mean {row['mean_days']:.0f}d",
            va="center", ha="left", fontsize=10, color=NEAR_BLACK)
ax.set_yticks(y, plot_steps["display"])
ax.set_xlim(0, max_x + 18)
ax.set_xlabel("Days")
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.spines["left"].set_color(GREY_AXIS)
ax.spines["bottom"].set_color(GREY_AXIS)
ax.annotate("Near-immediate handoff", xy=(plot_steps.loc[1, 'median_days'], 1), xytext=(20, 1.35),
            textcoords='data', arrowprops=dict(arrowstyle='-|>', color=GREY_AXIS, lw=1.1),
            fontsize=9.5, color=MID_GREY)
ax.annotate("Credible reform target", xy=(plot_steps.loc[0, 'median_days'], 0), xytext=(27, -0.35),
            textcoords='data', arrowprops=dict(arrowstyle='-|>', color=MAROON, lw=1.2),
            fontsize=9.5, color=MAROON)

save_polished(fig, RESOLUTION_FILES["routing_steps"])

fig, ax = plt.subplots(figsize=(9.2, 5.0))
fig.subplots_adjust(left=0.18, right=0.97, top=0.96, bottom=0.20)
labels = ["Current Route 4", "Route 2 benchmark"]
values = [route_counterfactual["current_median"], route_counterfactual["benchmark_median"]]
colors = [MAROON, GOOD_GREEN]
bars = ax.barh(labels, values, color=colors, height=0.48)
for bar, val in zip(bars, values):
    ax.text(val + 1.2, bar.get_y() + bar.get_height() / 2, f"{val:.0f}d", va="center", ha="left",
            fontsize=12, color=NEAR_BLACK, fontweight="bold")
ax.set_xlim(0, max(values) + 18)
ax.set_xlabel("Median days")
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.spines["left"].set_color(GREY_AXIS)
ax.spines["bottom"].set_color(GREY_AXIS)
ax.text(
    (route_counterfactual["current_median"] + route_counterfactual["benchmark_median"]) / 2 + 2,
    0.5,
    f"Upper-bound saving ≈ {route_counterfactual['upper_bound_saved']:.0f}d",
    fontsize=10.5, color=NEAR_BLACK,
    va='center', ha='left',
    bbox=dict(boxstyle="round,pad=0.25", fc="#FFF5D8", ec="none"),
)

save_polished(fig, RESOLUTION_FILES["routing_upper_bound"])

prompt_reference = {
    "Disposed-only median": (64, 41),
    "Median discard time": (3, 3),
    "Disposed with benefit (%)": (8.45, 3.66),
    "Open cases": (18409, 107206),
    "Route 2 share / median": (78, 24),
    "Route 4 share / median": (20, 49),
}

derived_reference = {
    "Disposed-only median": (round(scorecard_metrics['disposed_median_2324']), round(scorecard_metrics['disposed_median_2425'])),
    "Median discard time": (round(scorecard_metrics['discard_median_2324']), round(scorecard_metrics['discard_median_2425'])),
    "Disposed with benefit (%)": (round(scorecard_metrics['benefit_pct_2324'], 2), round(scorecard_metrics['benefit_pct_2425'], 2)),
    "Open cases": (int(scorecard_metrics['open_count_2324']), int(scorecard_metrics['open_count_2425'])),
    "Route 2 share / median": (round(route2_summary['share_pct']), round(route2_summary['median_days'])),
    "Route 4 share / median": (round(route4_summary['share_pct']), round(route4_summary['median_days'])),
}

mismatches = []
for key, prompt_vals in prompt_reference.items():
    derived_vals = derived_reference[key]
    if derived_vals != prompt_vals:
        mismatches.append(f"- {key}: prompt {prompt_vals} vs notebook-derived {derived_vals}")

summary_md = f"""
### Resolution execution summary

Generated files:
- `{RESOLUTION_FILES['scorecard'].name}`
- `{RESOLUTION_FILES['status_shift'].name}`
- `{RESOLUTION_FILES['status_shift_pie'].name}`
- `{RESOLUTION_FILES['regression'].name}`
- `{RESOLUTION_FILES['open_monthly'].name}`
- `{RESOLUTION_FILES['routing_compare'].name}`
- `{RESOLUTION_FILES['routing_steps'].name}`
- `{RESOLUTION_FILES['routing_upper_bound'].name}`

Logic reused:
- Status grouping and yearly composition adapted from `status.ipynb` using the July-June `custom_year` convention
- Rural-housing regression reuse from the existing `1.04` regression output table
- PMAY routing and step-gap construction adapted from `actionhistory.ipynb` using the July-June `custom_year` convention

Key derived values used on-chart:
- Disposed-only median: {scorecard_metrics['disposed_median_2324']:.0f}d → {scorecard_metrics['disposed_median_2425']:.0f}d
- Median discard time: {scorecard_metrics['discard_median_2324']:.0f}d → {scorecard_metrics['discard_median_2425']:.0f}d
- Disposed with benefit: {scorecard_metrics['benefit_pct_2324']:.2f}% → {scorecard_metrics['benefit_pct_2425']:.2f}%
- Open cases: {int(scorecard_metrics['open_count_2324']):,} → {int(scorecard_metrics['open_count_2425']):,}
- Route 2 benchmark: {route2_summary['share_pct']:.2f}% of cases, median {route2_summary['median_days']:.2f}d
- Route 4 benchmark: {route4_summary['share_pct']:.2f}% of cases, median {route4_summary['median_days']:.2f}d
- Dominant route-4 pattern share within length-4 cases: {route4_pattern_share:.2f}%

Prompt mismatches flagged:
{chr(10).join(mismatches) if mismatches else '- None; prompt values matched after rounding.'}

Recommended insertion order:
1. `resolution_scorecard.png`
2. `resolution_status_shift.png`
3. `resolution_status_shift_pie.png`
4. `resolution_open_monthly_trend.png`
5. Existing disposed-time figure(s)
6. `regression_block_vs_district.png`
7. `routing_route2_vs_route4.png`
8. `routing_step_decomposition.png`
9. `routing_counterfactual_upper_bound.png`
"""

display(Markdown(summary_md))



---
## Summary of outputs

| Output | File | Notes |
|--------|------|-------|
| Petitioner frequency table | `output/tables/petitioner_frequency_distribution.xlsx` | Shares + cumulative shares |
| Repeat-filer subcategory | `output/tables/repeat_filer_subcategory_analysis.xlsx` | Same vs different subcategory |
| Disposal time histograms | `output/disposal_time_histogram.png`, `disposal_time_overlaid.png` | Disposed only, capped at 180d |
| Rural housing regression | `output/tables/rural_housing_regression.xlsx` | M1/M2/M3 comparison + block coefs |
| Resolution scorecard | `output/resolution_scorecard.png` | Policy-reader summary of improvement vs hidden deterioration |
| Status composition shift | `output/resolution_status_shift.png` | 2023-24 vs 2024-25 closure composition |
| Status composition shift (pie) | `output/resolution_status_shift_pie.png` | Two-donut variant of the yearly status mix using the same publication palette |
| Regression takeaway figure | `output/regression_block_vs_district.png` | R² comparison highlighting block vs district FE |
| Monthly open-count trend | `output/resolution_open_monthly_trend.png` | Monthly created-on counts for complaints grouped into Open |
| Routing comparison | `output/routing_route2_vs_route4.png` | Structured comparison of the short and long PMAY closure paths |
| Routing step decomposition | `output/routing_step_decomposition.png` | Mean/median delay by Route 4 step |
| Routing upper-bound counterfactual | `output/routing_counterfactual_upper_bound.png` | Descriptive upper-bound benchmark, explicitly non-causal |
